# FaultSense — Training Pipeline

Pipeline cascade (binary failure detection → multiclass failure type) untuk
dataset AI4I 2020. Memperbaiki dua masalah metodologi pada notebook lama:

1. **Threshold tuning di-pindah ke validation set** (sebelumnya di test set
   → data leakage).
2. **Hapus duplikasi balancing** pada model binary: training pakai SMOTE
   saja, tidak ditambah `class_weight="balanced"` lagi.

Output artefak yang dipakai oleh `app/prediction.py`:

- `models/scaler.pkl`
- `models/model_binary_rf.pkl`
- `models/model_multiclass_rf.pkl`
- `models/ohe_columns.pkl`
- `models/threshold.pkl`
- `models/label_map.pkl`

## Import library

In [ ]:
import os
import pickle
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Setup MLflow

Tracking ke SQLite store. **Tidak** auto-spawn UI — jalankan manual saat
perlu: `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5002`.

In [ ]:
db_path = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("FaultSense")
print("Tracking URI:", f"sqlite:///{db_path}")

## Konfigurasi path

In [ ]:
# Bila notebook dijalankan dari folder `notebook/`, naik satu level ke root
# repo. Bila dijalankan dari root, gunakan cwd langsung.
if os.path.basename(os.getcwd()) == "notebook":
    BASE_DIR = os.path.abspath("..")
else:
    BASE_DIR = os.path.abspath(".")

DATA_PATH  = os.path.join(BASE_DIR, "data", "ai4i2020.csv")
MODEL_DIR  = os.path.join(BASE_DIR, "models")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("BASE_DIR :", BASE_DIR)
print("DATA_PATH:", DATA_PATH)

## Load data dan EDA singkat

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Total baris: {len(df)}")

failure_types = ["TWF", "HDF", "PWF", "OSF", "RNF"]

print("\nDistribusi Machine failure:")
print(df["Machine failure"].value_counts())
print((df["Machine failure"].value_counts(normalize=True) * 100).round(2))

print("\nDistribusi failure type:")
for ft in failure_types:
    n = int(df[ft].sum())
    print(f"  {ft}: {n} ({n / len(df) * 100:.2f}%)")

# Catatan dataset: RNF=1 mayoritas justru punya Machine failure=0.
mf_with_rnf  = int(((df["Machine failure"] == 1) & (df["RNF"] == 1)).sum())
rnf_total    = int(df["RNF"].sum())
print(f"\nRNF=1 total: {rnf_total} (di antaranya hanya {mf_with_rnf} dengan Machine failure=1)")

## Preprocessing

Drop kolom identifier dan lakukan one-hot encoding pada kolom kategorikal
`Type` (H/L/M).

In [ ]:
# Kolom-kolom yang tidak relevan untuk model. `timestamp` muncul di varian
# dataset yang sudah diaugmentasi tanggal — drop bila ada, abaikan bila tidak.
drop_cols = [c for c in ["UDI", "Product ID", "timestamp"] if c in df.columns]
df_processed = df.drop(columns=drop_cols).copy()
df_processed = pd.get_dummies(df_processed, columns=["Type"], drop_first=False)
for col in ["Type_H", "Type_L", "Type_M"]:
    df_processed[col] = df_processed[col].astype(int)

print("Kolom setelah OHE:", df_processed.columns.tolist())

## Split train / val / test (60/20/20)

Test set tidak akan disentuh selama training & threshold tuning agar
evaluasi akhir benar-benar mengukur generalisasi.

In [ ]:
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_H",
    "Type_L",
    "Type_M",
]

X = df_processed[feature_cols]
y = df_processed["Machine failure"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# 0.25 dari 0.80 = 0.20 dari total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")
print(f"Train failure rate: {y_train.mean():.4f}")
print(f"Val   failure rate: {y_val.mean():.4f}")
print(f"Test  failure rate: {y_test.mean():.4f}")

## Scaling

`StandardScaler` di-fit hanya di train set untuk menghindari kebocoran
statistik dari val/test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=feature_cols, index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=feature_cols, index=X_test.index
)

## SMOTE (hanya pada training set)

SMOTE diterapkan setelah scaling, hanya pada training set, untuk menangani
imbalance. Val/test tetap natural agar metrik realistis.

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"Sebelum SMOTE: {dict(pd.Series(y_train).value_counts())}")
print(f"Setelah SMOTE: {dict(pd.Series(y_train_resampled).value_counts())}")

## Multiclass labels

Hanya gunakan baris dengan `Machine failure == 1` untuk training stage 2.
Penanganan kuirk dataset secara eksplisit:

- Baris failure tanpa failure type spesifik → **drop** dengan log jumlah.
- Baris dengan beberapa failure type aktif → **resolve** berdasarkan
  prioritas urutan `failure_types`.

In [ ]:
label_map = {ft: i for i, ft in enumerate(failure_types)}  # str → int


def get_failure_label(row: pd.Series) -> int | None:
    active = [ft for ft in failure_types if row[ft] == 1]
    if not active:
        return None
    return label_map[active[0]]


def build_multiclass_split(name: str, X_scaled: pd.DataFrame, y_binary: pd.Series):
    failure_idx = y_binary[y_binary == 1].index
    failure_df  = df_processed.loc[failure_idx, failure_types]

    no_label    = int((failure_df.sum(axis=1) == 0).sum())
    multi_label = int((failure_df.sum(axis=1) > 1).sum())
    print(f"[{name}] failure rows: {len(failure_idx)} | dropped (no label): {no_label} | multi-label: {multi_label}")

    y_multi = failure_df.apply(get_failure_label, axis=1).dropna().astype(int)
    X_multi = X_scaled.loc[y_multi.index]
    return X_multi, y_multi


X_train_multi, y_train_multi = build_multiclass_split("train", X_train_scaled, y_train)
X_val_multi,   y_val_multi   = build_multiclass_split("val",   X_val_scaled,   y_val)
X_test_multi,  y_test_multi  = build_multiclass_split("test",  X_test_scaled,  y_test)

print("\nDistribusi failure type (train multiclass):")
print(y_train_multi.value_counts().sort_index().rename(lambda i: failure_types[i]))

## Optuna tuning — binary

`class_weight` **tidak** dimasukkan sebagai parameter tuning karena dataset
train sudah di-rebalance via SMOTE; menambahkannya akan menggandakan efek
rebalancing.

In [ ]:
def objective_binary(trial: optuna.Trial) -> float:
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"         : trial.suggest_int("max_depth", 5, 15),
        "min_samples_split" : trial.suggest_int("min_samples_split", 5, 20),
        "min_samples_leaf"  : trial.suggest_int("min_samples_leaf", 2, 10),
        "max_features"      : trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state"      : 42,
        "n_jobs"            : -1,
    }
    model = RandomForestClassifier(**params)
    cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    y_pred = cross_val_predict(
        model, X_train_resampled, y_train_resampled, cv=cv
    )
    recall    = recall_score(y_train_resampled, y_pred, pos_label=1, zero_division=0)
    precision = precision_score(y_train_resampled, y_pred, pos_label=1, zero_division=0)
    return 0.85 * recall + 0.15 * precision


study_binary = optuna.create_study(direction="maximize", study_name="binary")
study_binary.optimize(objective_binary, n_trials=30, show_progress_bar=False)
print(f"Best CV score : {study_binary.best_value:.4f}")
print(f"Best params   : {study_binary.best_params}")

best_params_binary = {**study_binary.best_params, "random_state": 42, "n_jobs": -1}
rf_binary = RandomForestClassifier(**best_params_binary)
rf_binary.fit(X_train_resampled, y_train_resampled)

## Threshold tuning — pada validation set

Sebelumnya threshold dipilih berdasarkan performa di test set, sekarang
dilakukan murni di validation set sehingga test set bersih untuk evaluasi
akhir.

In [ ]:
y_val_prob = rf_binary.predict_proba(X_val_scaled)[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)
threshold_records = []
for t in thresholds:
    y_pred_t = (y_val_prob >= t).astype(int)
    threshold_records.append(
        {
            "threshold" : float(t),
            "recall"    : recall_score(y_val, y_pred_t, pos_label=1, zero_division=0),
            "precision" : precision_score(y_val, y_pred_t, pos_label=1, zero_division=0),
            "f1"        : f1_score(y_val, y_pred_t, pos_label=1, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_records)
MIN_PRECISION = 0.30
candidates = threshold_df[threshold_df["precision"] >= MIN_PRECISION]
if len(candidates) > 0:
    best_row = candidates.sort_values(["recall", "f1"], ascending=False).iloc[0]
    strategy = f"max recall dgn precision >= {MIN_PRECISION}"
else:
    best_row = threshold_df.sort_values(["recall", "f1"], ascending=False).iloc[0]
    strategy = "fallback: max recall tanpa batasan precision"

best_threshold = float(best_row["threshold"])
print(f"Strategi : {strategy}")
print(f"Threshold: {best_threshold:.2f}")
print(f"Val recall   : {best_row['recall']:.4f}")
print(f"Val precision: {best_row['precision']:.4f}")
print(f"Val F1       : {best_row['f1']:.4f}")

## Optuna tuning — multiclass

Multiclass tetap pakai `class_weight` karena distribusi failure type
sangat tidak seimbang dan SMOTE tidak diterapkan di sini.

In [ ]:
def objective_multiclass(trial: optuna.Trial) -> float:
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"         : trial.suggest_int("max_depth", 3, 25),
        "min_samples_split" : trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf"  : trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features"      : trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight"      : trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]),
        "random_state"      : 42,
        "n_jobs"            : -1,
    }
    model = RandomForestClassifier(**params)
    cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    y_pred = cross_val_predict(model, X_train_multi, y_train_multi, cv=cv)
    return f1_score(y_train_multi, y_pred, average="macro", zero_division=0)


study_multiclass = optuna.create_study(direction="maximize", study_name="multiclass")
study_multiclass.optimize(objective_multiclass, n_trials=30, show_progress_bar=False)

best_params_multi = {**study_multiclass.best_params, "random_state": 42, "n_jobs": -1}
rf_multiclass = RandomForestClassifier(**best_params_multi)
rf_multiclass.fit(X_train_multi, y_train_multi)
print(f"Best multiclass CV F1-macro: {study_multiclass.best_value:.4f}")

## Evaluasi akhir di test set

In [ ]:
y_test_prob   = rf_binary.predict_proba(X_test_scaled)[:, 1]
y_test_pred   = (y_test_prob >= best_threshold).astype(int)

acc       = accuracy_score(y_test, y_test_pred)
recall    = recall_score(y_test, y_test_pred, pos_label=1, zero_division=0)
precision = precision_score(y_test, y_test_pred, pos_label=1, zero_division=0)
f1        = f1_score(y_test, y_test_pred, pos_label=1, zero_division=0)
auc       = roc_auc_score(y_test, y_test_prob)

print("=" * 60)
print("        EVALUASI FINAL — TEST SET")
print("=" * 60)
print(f"Binary    | acc={acc:.4f}  recall={recall:.4f}  precision={precision:.4f}  f1={f1:.4f}  AUC={auc:.4f}")

if len(y_test_multi) > 0:
    y_test_pred_multi = rf_multiclass.predict(X_test_multi)
    all_class_labels  = sorted(label_map.values())
    f1_macro          = f1_score(
        y_test_multi, y_test_pred_multi,
        average="macro", labels=all_class_labels, zero_division=0,
    )
    print(f"Multiclass| F1-macro={f1_macro:.4f}")
    print()
    print("--- Classification report (binary) ---")
    print(classification_report(y_test, y_test_pred, target_names=["Normal", "Failure"], zero_division=0))
    print("--- Classification report (multiclass) ---")
    print(classification_report(
        y_test_multi, y_test_pred_multi,
        labels=all_class_labels, target_names=failure_types, zero_division=0,
    ))
else:
    f1_macro = float("nan")
    print("Multiclass| (test set tidak punya failure rows)")

## Simpan artefak untuk API

In [ ]:
ohe_columns = ["Type_H", "Type_L", "Type_M"]
# label_map untuk inference: int → str (kebalikan dari label_map training).
label_map_inference = {v: k for k, v in label_map.items()}
# Filter hanya kelas yang dipelajari oleh model agar API tidak menjanjikan
# label yang tak pernah bisa muncul.
trained_classes = set(int(c) for c in rf_multiclass.classes_)
label_map_inference = {
    cls: label_map_inference[cls] for cls in trained_classes if cls in label_map_inference
}
print("label_map yang disimpan (int → nama):", label_map_inference)

artifacts = {
    "scaler.pkl"             : scaler,
    "model_binary_rf.pkl"    : rf_binary,
    "model_multiclass_rf.pkl": rf_multiclass,
    "ohe_columns.pkl"        : ohe_columns,
    "threshold.pkl"          : best_threshold,
    "label_map.pkl"          : label_map_inference,
}
for name, obj in artifacts.items():
    path = os.path.join(MODEL_DIR, name)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  saved {name} ({os.path.getsize(path) / 1024:.1f} KB)")

## Logging ke MLflow

In [ ]:
with mlflow.start_run(run_name="final"):
    mlflow.log_params(
        {
            **{f"binary_{k}": v for k, v in best_params_binary.items()},
            **{f"multi_{k}":  v for k, v in best_params_multi.items()},
            "best_threshold"     : best_threshold,
            "threshold_strategy" : strategy,
            "min_precision"      : MIN_PRECISION,
        }
    )
    mlflow.log_metrics(
        {
            "test_accuracy"           : float(acc),
            "test_recall"             : float(recall),
            "test_precision"          : float(precision),
            "test_f1"                 : float(f1),
            "test_auc"                : float(auc),
            "test_f1_macro_multiclass": float(f1_macro) if not np.isnan(f1_macro) else 0.0,
            "val_recall"              : float(best_row["recall"]),
            "val_precision"           : float(best_row["precision"]),
            "val_f1"                  : float(best_row["f1"]),
        }
    )
    mlflow.sklearn.log_model(rf_binary,     "model_binary_rf")
    mlflow.sklearn.log_model(rf_multiclass, "model_multiclass_rf")
    mlflow.log_artifact(os.path.join(MODEL_DIR, "threshold.pkl"))
    mlflow.log_artifact(os.path.join(MODEL_DIR, "label_map.pkl"))
    print("Logged ke MLflow.")

print("\nDone.")